# ShopTalk-X — Day 3: Multimodal (captions + CLIP) image search demo

Checkpoint per the execution plan: **photo search working end-to-end**.

Prerequisites (run once, in order):
```
python -m shoptalk.data.caption_images     # BLIP captions -> products.parquet
python -m shoptalk.embeddings.embed_text   # re-embed caption-augmented documents
python -m shoptalk.embeddings.embed_image  # CLIP-embed catalog images -> image collection
```

This notebook queries the system with a **photo** instead of typed text:
photo -> CLIP embedding -> ANN over the image collection -> BLIP captions the
*query photo itself* (pseudo-text query) -> the same Day-2 cross-encoder
reranks candidates against that pseudo-query. See
`src/shoptalk/retrieval/image_search.py` for why: the cross-encoder is
text-only, and a photo has no natural-language query to pair it with, so we
manufacture one from the photo rather than build/maintain a second reranker.

In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
from PIL import Image

from shoptalk.retrieval.image_search import image_search
from shoptalk.config import load_config

cfg = load_config()
print("image collection:", cfg["clip"]["collection_name"])
print("CLIP model:", cfg["clip"]["model_name"], cfg["clip"]["pretrained"])

## Pick a query photo

For a real "in-the-wild" demo, point `QUERY_IMAGE` at any photo on disk (a
picture you took of a similar product). As a smoke test with no external
photo handy, this defaults to one of the catalog's own images -- expect the
exact matching product to come back at rank 1 with CLIP similarity ~1.0,
which is a useful correctness check even though it isn't a fair "recall"
test (see the writeup at the bottom for that).

In [ ]:
import pandas as pd

products = pd.read_parquet("../data/processed/products.parquet")
sample = products[products["image_available"]].sample(1, random_state=7).iloc[0]
QUERY_IMAGE = sample["image_path"]

print("query image:", QUERY_IMAGE)
print("(ground-truth product, for reference only):", sample["item_name"])
Image.open(QUERY_IMAGE).convert("RGB")

## Run the two-stage image search pipeline

In [ ]:
result = image_search(QUERY_IMAGE, stage1_k=30, top_k=5, cfg=cfg)

print("BLIP pseudo-query (used for cross-encoder rerank):", repr(result["pseudo_query"]))
hits = result["hits"]
pd.DataFrame(
    [
        {
            "rank": i + 1,
            "item_name": h["metadata"]["item_name"][:60],
            "category": h["metadata"]["category"],
            "price_usd": h["metadata"]["price_usd"],
            "clip_score": round(h["stage1_score"], 3),
            "rerank_score": round(h["rerank_score"], 3),
            "item_id": h["item_id"],
        }
        for i, h in enumerate(hits)
    ]
)

## Visual check: query photo vs. top-K results

In [ ]:
fig, axes = plt.subplots(1, len(hits) + 1, figsize=(3 * (len(hits) + 1), 3.5))

axes[0].imshow(Image.open(QUERY_IMAGE).convert("RGB"))
axes[0].set_title("query photo", fontsize=9)
axes[0].axis("off")

for ax, hit in zip(axes[1:], hits):
    img_path = hit["metadata"]["image_path"]
    ax.imshow(Image.open(img_path).convert("RGB"))
    ax.set_title(f"#{hits.index(hit)+1} rerank={hit['rerank_score']:.2f}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Notes

- **CLIP similarity vs. rerank score are different scales** -- CLIP scores are
  cosine similarity in [-1, 1]; cross-encoder rerank scores are unbounded
  logits. Never compare them directly across stages, only within a stage's
  own ranking (same pattern as Day 2's `stage1_score` vs `rerank_score`).
- **Fair evaluation of image search** (as opposed to this correctness smoke
  test) needs query photos that are *not* already catalog images -- e.g. the
  Day 5 robustness set (blur/occlusion/glare augmentations of catalog photos)
  or genuinely new photos of the same products from another angle. Recall@K
  for the image-query path should be reported separately from the text-query
  path in the Day-6/7 evaluation writeup.
- If `image_search` returns poor pseudo-queries for cluttered "in-the-wild"
  photos (multiple objects, background clutter), that's exactly the gap the
  Day-3/CV-depth YOLO detect-and-crop step (design doc §3.3a) is meant to
  close -- crop to the product before CLIP-embedding, rather than embedding
  the whole scene.